In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import sklearn


In [ ]:
from sklearn.metrics import(
    roc_auc_score,
    f1_score,
    precision_score,
    accuracy_score,
    recall_score
)

In [ ]:
from torchvision.models import densenet121

In [ ]:
device = torch.device("cuda")

model = densenet121(weights=None)

model.classifier = nn.Linear(
    model.classifier.in_features,
    14
)

model = model.to(device)

model.load_state_dict(
    torch.load(
        "D:/Chest_XRay_Project/models/best_model.pth"
    )
)


In [ ]:
from PIL import Image
from torchvision import transforms

transform = transforms.Compose ([
 transforms.Resize((224,224)),
 transforms.Grayscale(num_output_channels=3),
 transforms.ToTensor()   
])

In [ ]:
test_list = pd.read_csv("D:/NIH chest X-Ray/test_list.txt",
                        header = None,
                        names = ["Image Index"])

import ast

df3=pd.read_csv('D:/Chest_XRay_Project/notebooks/result.csv')
df3["Labels_encoded"] = df3["Labels_encoded"].apply(ast.literal_eval)


test_df = df3[
    df3["Image Index"].isin(test_list["Image Index"])
]

In [ ]:
from torch.utils.data import Dataset



class XrayDataset(Dataset):
    def __init__(self,df,transform=None):
        self.df = df.reset_index(drop = True)
        self.transform = transform 
        
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self,idx):
        row = self.df.iloc[idx]
        
        img_path = row["Image Path"]
        label = row["Labels_encoded"]
        
        image = Image.open(img_path)
        
        if self.transform:
            image = self.transform(image)
        
        label = torch.tensor(label, dtype=torch.float32)
            
        return image, label

In [ ]:
from torch.utils.data import DataLoader

test_dataset = XrayDataset(test_df, transform = transform)

test_dataloader = DataLoader(test_dataset, batch_size = 32, shuffle = True, num_workers = 0, pin_memory=True )


In [ ]:
all_labels = []
all_outputs = []

model.eval()

with torch.inference_mode():
    
    for images,batch_labels in test_dataloader:
        
        images = images.to(device)
        batch_labels = batch_labels.to(device)
        
        outputs = model(images)
        
        all_outputs.append(outputs.cpu())
        all_labels.append(batch_labels.cpu())
        
        

In [ ]:
all_labels = torch.cat(all_labels)
all_outputs = torch.cat(all_outputs)
# Convert logits to probabilities
probabilities = torch.sigmoid(all_outputs)

# Convert to numpy
probabilities = probabilities.cpu().numpy()
labels = all_labels.numpy()


In [ ]:



thresholds = np.arange(0.01, 1.0, 0.01)

best_thresholds = []

for disease in range(14):

    best_f1 = 0
    best_threshold = 0.5

    for threshold in thresholds:

        predictions = probabilities[:, disease] > threshold
        

        f1 = f1_score(
            labels[:, disease],
            predictions
        )

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    best_thresholds.append(best_threshold)
    
    print(
        f"Disease {disease}: "
        f"Best threshold={best_threshold:.2f}, "
        f"F1={best_f1:.3f}"
    )


In [ ]:

import json

with open(r"D:\Chest_XRay_Project\notebooks\disease_mapping.json", "r") as f:
    disease_mapping = json.load(f)


threshold_data = {}

for disease, threshold in zip(disease_mapping, best_thresholds):
    threshold_data[disease] = float(threshold)

with open(
    "D:/Chest_XRay_Project/models/best_threshold.json",
    "w"
) as f:
    json.dump(threshold_data, f, indent=4)

In [ ]:
best_thresholds

In [ ]:
print(probabilities.shape)
print(np.array(best_thresholds).shape)

In [ ]:
y_pred = (probabilities > best_thresholds).astype(int)

In [ ]:
y_pred.shape

In [ ]:
y_true = all_labels.numpy()
y_prob = probabilities


In [ ]:
print(y_true.shape)
print(y_pred.shape)
print(probabilities.shape)

In [ ]:
f1 = f1_score(
    y_true,
    y_pred,
    average = "macro"
)

precision = precision_score(
    y_true,
    y_pred,
    average = "macro"
)

recall = recall_score(
    y_true,
    y_pred,
    average = "macro"
    
)

auroc = roc_auc_score(
    y_true,
    y_prob,
    average = "macro"
)

accuracy = accuracy_score(
    y_true,
    y_pred,
)


In [ ]:
print("F1:", f1)
print("Precision:", precision)
print("Recall:", recall)
print("AUROC:", auroc)
print("Accuracy:", accuracy)